# Uniform Int4 — Master Runbook

**Project**: `parameter-golf-uniform-int4`  
**Target**: OpenAI Parameter Golf Challenge (8×H100, <10min, 16MB artifact)  
**Proven Baseline**: 1.453 BPB (T4 @ 32k batch, 3000 steps, 2026-04-09)

---

## Instructions
1. Select your runtime: `Runtime > Change runtime type > GPU (T4) or TPU (v5e)`
2. Run All: `Runtime > Run all`
3. Cell 3 is a smoke test (~60s). If it prints `val_bpb`, the pipeline works.
4. Cell 4 is the full run. Monitor step logs for heartbeat.

## Hardware Notes
| Runtime | Batch Tokens | Steps for ~1.45 BPB | Time |
|---------|-------------|---------------------|------|
| T4 GPU | 32,768 | 3,000 | ~3.7h |
| TPU v5e | 262,144 (test) | ~375 | ~45min (est) |
| 8×H100 (leaderboard) | 524,288 | 20,000 | <10min |

In [ ]:
# [CELL 1] CLONE REPO
%cd /content
!rm -rf uniform-int4
!git clone https://github.com/jmoncayo-pursuit/parameter-golf-uniform-int4.git uniform-int4
print('✅ Repo cloned')

In [ ]:
# [CELL 2] DOWNLOAD TRAINING ASSETS (~1.5GB)
%cd /content/uniform-int4
!mkdir -p data/tokenizers data/datasets/fineweb10B_sp1024

!wget -q -O data/tokenizers/fineweb_1024_bpe.model \
  https://huggingface.co/datasets/willdepueoai/parameter-golf/resolve/main/datasets/tokenizers/fineweb_1024_bpe.model
!wget -q -O data/datasets/fineweb10B_sp1024/fineweb_val_000000.bin \
  https://huggingface.co/datasets/willdepueoai/parameter-golf/resolve/main/datasets/datasets/fineweb10B_sp1024/fineweb_val_000000.bin
!wget -q -O data/datasets/fineweb10B_sp1024/fineweb_train_000001.bin \
  https://huggingface.co/datasets/willdepueoai/parameter-golf/resolve/main/datasets/datasets/fineweb10B_sp1024/fineweb_train_000001.bin

!pip install -q sentencepiece zstandard PyYAML huggingface-hub datasets tqdm

# Verify assets
import os
tok = os.path.getsize('data/tokenizers/fineweb_1024_bpe.model')
val = os.path.getsize('data/datasets/fineweb10B_sp1024/fineweb_val_000000.bin')
trn = os.path.getsize('data/datasets/fineweb10B_sp1024/fineweb_train_000001.bin')
print(f'Tokenizer: {tok:,} bytes')
print(f'Val shard: {val:,} bytes')
print(f'Train shard: {trn:,} bytes')
assert tok > 100_000, '❌ Tokenizer download failed'
assert val > 1_000_000, '❌ Val shard download failed'
assert trn > 100_000_000, '❌ Train shard download failed'
print('✅ All assets verified')

In [ ]:
# [CELL 3] SMOKE TEST — warmup + 1 step + validation (~60s)
# If this prints a val_bpb score, the full pipeline is proven.
%cd /content/uniform-int4
!env NO_COMPILE=1 ITERATIONS=1 python3 train_gpt.py

In [ ]:
# [CELL 4] FULL TRAINING RUN
# Defaults are baked into train_gpt.py (32k batch, 3000 steps, 1024 seq).
# Override via env vars for different hardware:
#   TRAIN_BATCH_TOKENS=262144 ITERATIONS=3000  (for larger GPU/TPU)
#   TRAIN_BATCH_TOKENS=524288 ITERATIONS=20000  (for 8xH100 leaderboard)
%cd /content/uniform-int4
!env NO_COMPILE=1 python3 train_gpt.py